# Memory management

In [ ]:
# **************************
# (example 1) simple history
# **************************

import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.messages import AIMessage, HumanMessage
from dotenv import load_dotenv
import os,warnings

In [ ]:
warnings.filterwarnings("ignore")

env = r"D:\stackroute\2_AI-assisted-programming\learning_requirements\bosch\2026\5_advancedPE\code\config.env"
if load_dotenv(env):
    gemini_key = os.getenv("GEMINI_API_KEY")

In [ ]:
# History is a list of messages. Each message is either a HumanMessage or an AIMessage.

history = InMemoryChatMessageHistory()

history.add_user_message("hello") # user's first message
history.add_ai_message("Greetings. How are you ?") # AI's reply

history.add_user_message("i am fine. today i want to talk about some interesting topics") # user message
history.add_ai_message("Nice. Looking forward to this.") # AI reply

print(history)

In [ ]:
# *******************************************
# (example 2) : Simulation of a real-life chat
# ********************************************
history = InMemoryChatMessageHistory()

# 1) Create Gemini model
llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", api_key=gemini_key, temperature=0)

# Start the chat loop
msg = '''   Hi! I am your AI Automotive Consultant. How can I help you today?
            (Ask about vehicle specs, maintenance tips, or buying advice)

            To end this conversation, type exit/bye/quit/close.
        '''

flag = True
exit_val = ["exit","quit","bye","close"]

print(msg)

In [ ]:
while flag:
    user_input = input("You: ")
    if any(ev for ev in exit_val if ev in user_input):
        flag = False

    # Add user message to history
    history.add_user_message(user_input)

    # Generate AI response using chat history
    ai_response = llm.invoke(history.messages)

    # Display and store AI response
    print(f"User: {user_input}")
    print(f"AI: {ai_response.content}")
    history.add_ai_message(ai_response.content)

In [ ]:
# total messages
print(f"There are {len(history.messages)} messages")

In [ ]:
# display the full chat history

for msg in history.messages:
    role = "User" if isinstance(msg, HumanMessage) else "AI"
    if role == "AI":
        message = msg.content[0]['text']
    else:
        message = msg.content

    print(f"{role}: {message}")

In [ ]:
# # display the full chat history
# print("\n--- Chat History ---")
# for msg in history.messages:
#     role = "User" if isinstance(msg, HumanMessage) else "AI"
#     print(f"{role}: {msg.content}")
    

In [ ]:
# Example 3)
# ----------

# A simple langchain example to simulate a Manual entity memory.
# Entity Memory in LangChain is a specialized memory module that extracts, stores, and updates facts about specific entities (people, places, organizations, objects) mentioned during a conversation.

In [ ]:
import pandas as pd, random
from google import genai
from langchain_core.runnables import RunnableLambda
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [ ]:
csv_path = r"D:\stackroute\2_AI-assisted-programming\learning_requirements\bosch\2026\5_advancedPE\dataset\household_energy_requirement.csv"
data = pd.read_csv(csv_path)

entity_id = data.entid.tolist()
client = genai.Client(api_key=gemini_key)

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [ ]:
data

In [ ]:
# The entity data is the memory data that comes from the CSV
# This data will be passed to the LLM for processing and generating responses based on the entity information.

entities = [ "entid", "house_type", "num_occupants", "has_ac", "num_ac_units", "daily_energy_consumption_kWh", 
            "monthly_energy_consumption_kWh","estimated_peak_load_kW"]

# Select a random entity ID from the list of entity IDs and extract the corresponding entity data from the DataFrame
entid = random.choice(entity_id)
ent_data = data[entities][data.entid == entid]

# Convert the output into a JSON format
ent_data = ent_data.iloc[0].to_dict()
print(ent_data)

In [ ]:

def call_gemini(prompt_value):
    # PromptTemplate produces a PromptValue object
    prompt_text = prompt_value.to_string()

    response = client.models.generate_content(model="gemini-3.1-flash-lite", contents=prompt_text)

    # Return an AIMessage for StrOutputParser
    return AIMessage(content=response.text)

gemini_runnable = RunnableLambda(call_gemini)

# Create a reusable prompt template
prompt = PromptTemplate(input_variables=["ent_data", "question"],
    template="""
You are an energy advisor.

Use the following stored entity memory:

{ent_data}

Question:
{question}

Answer in simple terms.
"""
)

# Form the chain by combining the prompt, Gemini LLM, and output parser
chain = prompt | gemini_runnable | StrOutputParser()

In [ ]:
# form a question to ask the LLM based on the entity data

query = "Is this household likely to have high energy consumption? Explain why"

response = chain.invoke({ "ent_data": ent_data, "question": query})

In [ ]:
print(response)